In [1]:

import os
import zipfile
import re
import pandas as pd
from tqdm import tqdm

In [2]:
# Paths
zip_folder = '../data/sample/'
extracted_base_folder = '../extracted_data'


In [3]:


# # Ensure the output folder exists
# os.makedirs(extracted_base_folder, exist_ok=True)

# # STEP 1: Extract all zip files and delete them afterward
# for zip_filename in os.listdir(zip_folder):
#     if zip_filename.endswith('.zip'):
#         zip_path = os.path.join(zip_folder, zip_filename)
#         extract_folder_name = zip_filename.replace('.zip', '')
#         extract_path = os.path.join(extracted_base_folder, extract_folder_name)

#         with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#             zip_ref.extractall(extract_path)

#         os.remove(zip_path)  # Delete zip file to save space

# # STEP 2: Helper Functions

def extract_cik(filename):
    match = re.search(r'_edgar_data_(\d+)_', filename)
    return match.group(1) if match else None

def extract_item_7(text):
    normalized_text = text.lower()
    pattern = r'[\r\n]+\s*item[\s\n]*7[\s\n]*\.?[\s\n]*\:?[\s\n]*-?[\s\n]*(.*?)(?=[\r\n]+\s*item[\s\n]*7[\s\n]*a[\s\n]*\.?[\s\n]*?-?|[\r\n]+\s*item[\s\n]*8[\s\n]*-?|\Z)'
    matches = list(re.finditer(pattern, normalized_text, re.DOTALL | re.IGNORECASE))
    sections = [text[m.start(1):m.end(1)].strip() for m in matches]
    return sections if sections else None

def extract_risk_factors(text):
    normalized_text = text.lower()
    pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*\:?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*(.*?)(?=[\r\n]+\s*item[\s\n]*1[\s\n]*b[\s\n]*\.?[\s\n]*?-?|[\r\n]+\s*item[\s\n]*2[\s\n]*-?|\Z)'
    matches = list(re.finditer(pattern, normalized_text, re.DOTALL | re.IGNORECASE))
    sections = [text[m.start(1):m.end(1)].strip() for m in matches]
    return sections if sections else None




In [4]:


# STEP 3: Define helper functions
def extract_cik(filename):
    match = re.search(r'_edgar_data_(\d+)_', filename)

    if match:
        cik = match.group(1)
        return cik
    else:
        print("CIK not found:",filename)

def extract_item_7(text):
    normalized_text = text.lower()

    # New pattern: anchors to line beginnings, and uses \b for better word boundary matching
    pattern = r'^\s*item\s+7\.?\s*(.*?)(?=^\s*item\s+7a\.?|^\s*item\s+8\b|^\s*item\s+\d+\s*\b|$\Z)'

    match = re.search(pattern, normalized_text, re.IGNORECASE | re.DOTALL | re.MULTILINE)

    if match:
        # Use the start and end index from the original text
        content_start, content_end = match.start(1), match.end(1)
        return [text[content_start:content_end].strip()]

    return None

def extract_item_7(text):
    normalized_text = text.lower()

    # New pattern: anchors to line beginnings, and uses \b for better word boundary matching
    pattern = r'[\r\n]+\s*item[\s\n]*7[\s\n]*\.?[\s\n]*\:?[\s\n]*-?[\s\n]*(.*?)(?=[\r\n]+\s*item[\s\n]*7[\s\n]*a[\s\n]*\.?[\s\n]*?-?|[\r\n]+\s*item[\s\n]*8[\s\n]*-?|\Z)'

    # Search for the pattern in the normalized text
    matches = list(re.finditer(pattern, normalized_text, re.DOTALL | re.IGNORECASE))

    # Extract the desired section if found
    sections = []
    for match in matches:
        content_start, content_end = match.start(1), match.end(1)
        sections.append(text[content_start:content_end].strip())

    # If the main pattern isn't found, check if "Item 1A. Risk Factors" appears standalone
    if not sections:
        standalone_pattern = r'[\r\n]+\s*item[\s\n]*7[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*'

        standalone_match = re.search(standalone_pattern, normalized_text)
        if standalone_match:
            sections.append(text[standalone_match.start(1):].strip())

    return sections if sections else None

def extract_risk_factors(text):
    # Normalize the text to handle different cases and line breaks
    normalized_text = text.lower()

    # Define the main regex pattern to extract the content between "Item 1A. Risk Factors" and the subsequent section
    pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*\:?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*(.*?)(?=[\r\n]+\s*item[\s\n]*1[\s\n]*b[\s\n]*\.?[\s\n]*?-?|[\r\n]+\s*item[\s\n]*2[\s\n]*-?|\Z)'

    # Search for the pattern in the normalized text
    matches = list(re.finditer(pattern, normalized_text, re.DOTALL | re.IGNORECASE))

    # Extract the desired section if found
    sections = []
    for match in matches:
        content_start, content_end = match.start(1), match.end(1)
        sections.append(text[content_start:content_end].strip())

    # If the main pattern isn't found, check if "Item 1A. Risk Factors" appears standalone
    if not sections:
        standalone_pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*'

        standalone_match = re.search(standalone_pattern, normalized_text)
        if standalone_match:
            sections.append(text[standalone_match.start(1):].strip())

    return sections if sections else None

import re

def extract_risk_factors(text):
    # Normalize the text to handle different cases and line breaks for the main pattern
    normalized_text = text.lower()

    # Define the main regex pattern to extract the content between "Item 1A. Risk Factors" and the subsequent section
    pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*(.*?)(?=[\r\n]+\s*item[\s\n]*1[\s\n]*b[\s\n]*\.?[\s\n]*?-?|[\r\n]+\s*item[\s\n]*2[\s\n]*-?|\Z)'

    # Search for the main pattern in the normalized text
    matches = list(re.finditer(pattern, normalized_text, re.DOTALL | re.IGNORECASE))
    
    # Extract the desired section if found
    sections = []
    for match in matches:
        content_start, content_end = match.start(1), match.end(1)
        sections.append(text[content_start:content_end].strip())
    
    # If the main pattern isn't found, check if "Item 1A. Risk Factors" appears standalone
    if not sections:
        standalone_pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*'
        standalone_match = re.search(standalone_pattern, normalized_text)
        if standalone_match:
            sections.append(text[standalone_match.start(1):].strip())
    
    # Add an alternative pattern for standalone "RISK FACTORS" in all caps, from its first occurrence to "Item 2. Properties"
    if not sections:
        # Modified pattern: ensures "RISK FACTORS" is not directly preceded by a quote (e.g., "RISK FACTORS")
        alt_pattern = r'(?<!")\bRISK\s*FACTORS\b[^\r\n]*[\s\n]*(.*?)(?=\bItem\s*2\.\s*Properties\b)'
        alt_match = re.search(alt_pattern, text, re.DOTALL | re.IGNORECASE)  # Case-insensitive match for "Item 2. Properties"
        
        if alt_match:
            content_start, content_end = alt_match.start(0), alt_match.end(0)
            section_text = text[content_start:content_end].strip()
            sections.append(section_text)
    
    return sections if sections else None

def full_content_extraction(text):
    match = re.search(r"table of contents(.*)", text, re.DOTALL | re.IGNORECASE)
    
    if match:
        extracted_text = match.group(1)
        return extracted_text
    else:
        return None

In [5]:
from tqdm import tqdm
import os

# Delete all files that are not .txt with '10-K' in filename
for root, _, files in os.walk(extracted_base_folder):
    for file in tqdm(files, desc=f"Processing files in {root}"):
        if not (file.endswith('.txt') and '10-K' in file):
            file_path = os.path.join(root, file)
            try:
                os.remove(file_path)
                # print(f"Deleted: {file_path}")
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")



Processing files in ../extracted_data: 0it [00:00, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000: 0it [00:00, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000\1993: 0it [00:00, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000\1993\QTR1: 0it [00:00, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000\1993\QTR2: 0it [00:00, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000\1993\QTR3: 0it [00:00, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000\1993\QTR4: 100%|██████████████████████████| 6/6 [00:00<?, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000\1994: 0it [00:00, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000\1994\QTR1: 100%|████████████████████| 1420/1420 [00:00<?, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000\1994\QTR2: 100%|██████████████████████| 521/521 [00:00<?, ?it/s]
Processing files in ../extracted_data\10-X_C_1993-2000\1994\QTR3: 100%|█████████████████████

In [6]:
# STEP 3: Collect all relevant .txt files

import pyarrow  # Needed for Parquet support with pandas
import time

data = []
batch_size = 10000  # Save every 100 files
batch_count = 0


txt_files = []
for root, _, files in os.walk(extracted_base_folder):
    for file in files:
        if file.endswith('.txt') and '10-K' in file:
            txt_files.append(os.path.join(root, file))


In [7]:
txt_files[-50:]

['../extracted_data\\10-X_C_2024\\2024\\QTR4\\20241220_10-K_edgar_data_1967097_0001967097-24-000011.txt',
 '../extracted_data\\10-X_C_2024\\2024\\QTR4\\20241220_10-K_edgar_data_1978811_0001558370-24-016433.txt',
 '../extracted_data\\10-X_C_2024\\2024\\QTR4\\20241220_10-K_edgar_data_356037_0000356037-24-000070.txt',
 '../extracted_data\\10-X_C_2024\\2024\\QTR4\\20241220_10-K_edgar_data_794170_0000794170-24-000051.txt',
 '../extracted_data\\10-X_C_2024\\2024\\QTR4\\20241220_10-K_edgar_data_936395_0000936395-24-000044.txt',
 '../extracted_data\\10-X_C_2024\\2024\\QTR4\\20241223_10-K-A_edgar_data_1043150_0001683168-24-008950.txt',
 '../extracted_data\\10-X_C_2024\\2024\\QTR4\\20241223_10-K-A_edgar_data_1393612_0001393612-24-000073.txt',
 '../extracted_data\\10-X_C_2024\\2024\\QTR4\\20241223_10-K-A_edgar_data_836147_0001437749-24-038297.txt',
 '../extracted_data\\10-X_C_2024\\2024\\QTR4\\20241223_10-K_edgar_data_1000230_0001437749-24-038248.txt',
 '../extracted_data\\10-X_C_2024\\2024\\QTR4

In [8]:
breach_list = pd.read_excel("../data/aa_plus_prc_breach_list_cik.xlsx")
breach_list

,cik,conm_br,date_breach,year_breach,source,cik_year
0,1468516,Aol Inc.,2004-06-23,2004,audit analytics,1468516_2004
1,104169,Sam's Club,2005-12-02,2005,audit analytics,104169_2005
2,1375557,GUIDANCE SOFTWARE INC,2005-12-20,2005,prc,1375557_2005
3,5907,AT&T CORP,2006-08-29,2006,prc,5907_2006
4,12978,OFFICEMAX INC,2006-02-09,2006,prc,12978_2006
...,...,...,...,...,...,...
737,1835268,Connect Biopharma Holdings Ltd,2023-04-11,2023,audit analytics,1835268_2023
738,1857475,Dole Plc,2023-03-22,2023,audit analytics,1857475_2023
739,1858681,"West Technology Group, LLC",2023-04-18,2023,audit analytics,1858681_2023
740,1875444,"Arhaus, Inc.",2023-02-15,2023,audit analytics,1875444_2023


In [9]:
breach_list.cik = breach_list.cik.astype(str)

In [10]:

# STEP 4: Extract and store data
data = []

for i, file_path in enumerate(tqdm(txt_files, desc="Processing files")):
    file_name = os.path.basename(file_path)
    cik = extract_cik(file_name)
    date_match = re.search(r'(\d{8})(?=_10-K)', file_name)
    date_str = date_match.group(1) if date_match else None
    date_formatted = pd.to_datetime(date_str, format='%Y%m%d').strftime('%Y-%m-%d') if date_str else None
    year = int(date_str[:4])
    if not cik:
        print("CIK not found:", file_name)
        continue
    if cik in breach_list.cik.unique():
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue
            
    

        item_7_sections = extract_item_7(text)
        risk_sections = extract_risk_factors(text)

        if (risk_sections is not None) and (item_7_sections is not None) and (len(risk_sections[-1])>10):
            full_content  = None

        else:
            full_content  = text

    
        text_lower = text.lower()
    
        # Check for presence of "item 1a"
        item_1a_flag = "item 1a" in text_lower
    
        # Check for "item 7" while avoiding "item 7a"
        item_7_flag = bool(re.search(r'\bitem\s+7\b(?!\s*a)', text_lower))  # "item 7" but not "item 7a"
        data.append({
            "cik": cik,
            "date": date_formatted,
            "file": file_name,
            "full_content": full_content,
            "item_7": item_7_sections,
            "risk_factors": risk_sections,
            "item_1a_flag": item_1a_flag,
            "item_7_flag": item_7_flag
        })
        # print(data)
        # Save every N files
        if (i + 1) % batch_size == 0:
            df = pd.DataFrame(data)
            batch_filename = f"parsed_batch_{batch_count}.parquet"
            df.to_parquet(f"../processed_data/new_{batch_filename}", index=False)
            print(f"✅ Saved batch {batch_count} to {batch_filename}")
            data = []  # Reset batch
            batch_count += 1

# Save any remaining data after loop
if data:
    df = pd.DataFrame(data)
    batch_filename = f"parsed_batch_{batch_count}_final.parquet"
    df.to_parquet(f"../processed_data/new_{batch_filename}", index=False)
    print(f"✅ Saved final batch to new_{batch_filename}")


# # Optional: Convert to DataFrame and export
# df = pd.DataFrame(data)
# df.to_csv("parsed_10K_sections.csv", index=False)

Processing files: 100%|███████████████████████████████████████████████████████| 298132/298132 [08:36<00:00, 577.76it/s]


✅ Saved final batch to new_parsed_batch_0_final.parquet


In [13]:
df[df.full_content.notna()].item_1a_flag.value_counts()

item_1a_flag
False    4414
True      681
Name: count, dtype: int64

In [17]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

In [18]:
merged_df = pd.merge(df,breach_list,on='cik')

In [22]:
result_df = merged_df[(merged_df.year_breach-merged_df.year<2) & (merged_df.year_breach>merged_df.year)]

In [23]:
result_df.to_parquet(f"../processed_data/merged_with_breach.parquet", index=False)